<a href="https://colab.research.google.com/github/natthamonsmn/Cinema_Team/blob/2/Cinema_function_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍿🎥✮⋆˙ ธุรกิจโรงภาพยนตร์ (Cinema)
โดยเน้น **ระบบจองที่นั่งดูหนัง** เช่น การจองที่นั่งของลูกค้า, ตรวจสอบที่นั่งว่าง, คำนวณราคาตามประเภทที่นั่งและประเภทสมาชิก

## ส่วนที่ 1 — class และฟังก์ชัน

In [1]:
from datetime import datetime, timedelta
import random
import pandas as pd
import numpy as np
import os, shutil
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

In [2]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

# 5. สร้างข้อมูลจำลองและแสดงผล
data = {
    'เมือง': ['กรุงเทพฯ', 'เชียงใหม่', 'ภูเก็ต'],
    'อุณหภูมิ': [30.5, 25.2, 28.9]
}

df_thai = pd.DataFrame(data)

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-tlwg-garuda fonts-tlwg-garuda-ttf fonts-tlwg-kinnari
  fonts-tlwg-kinnari-ttf fonts-tlwg-laksaman fonts-tlwg-laksaman-ttf
  fonts-tlwg-loma fonts-tlwg-loma-ttf fonts-tlwg-mono fonts-tlwg-mono-ttf
  fonts-tlwg-norasi fonts-tlwg-norasi-ttf fonts-tlwg-purisa
  fonts-tlwg-purisa-ttf fonts-tlwg-sawasdee fonts-tlwg-sawasdee-ttf
  fonts-tlwg-typewriter fonts-tlwg-typewriter-ttf fonts-tlwg-typist
  fonts-tlwg-typist-ttf fonts-tlwg-typo fonts-tlwg-typo-ttf fonts-tlwg-umpush
  fonts-tlwg-umpush-ttf fonts-tlwg-waree fonts-tlwg-waree-ttf
The following NEW packages will be installed:
  fonts-thai-tlwg fonts-tlwg-garuda fonts-tlwg-garuda-ttf fonts-tlwg-kinnari
  fonts-tlwg-kinnari-ttf fonts-tlwg-laksaman fonts-tlwg-laksaman-ttf
  fonts-tlwg-loma fonts-tlwg-loma-ttf fonts-tlwg-mono fonts-tlwg-mono-ttf
  fonts-tlwg-norasi fonts-tlwg-norasi-ttf fo

In [3]:
class Customer:
    def __init__(self, customer_id, name, member_type="General"):
        self.customer_id = customer_id
        self.name = name
        self.member_type = member_type
        self.points = 0

    def get_discount_rate(self, promo_discount_rate=0.0):
        member_discounts = {"General": 0.0, "Student": 0.20, "VIP": 0.30}
        member_rate = member_discounts.get(self.member_type, 0.0)
        return max(member_rate, promo_discount_rate)

    def add_points(self, net_price):
        earned = int(net_price // 10)
        self.points += earned
        return earned


class Showtime:
    def __init__(self, showtime_id, movie_name, theater_no, start_time, end_time, base_price):
        self.showtime_id = showtime_id
        self.movie_name = movie_name
        self.theater_no = theater_no
        self.start_time = start_time
        self.end_time = end_time
        self.base_price = base_price


class Booking:
    def __init__(self, booking_id, customer, showtime, seat_number, seat_type):
        self.booking_id = booking_id
        self.customer = customer
        self.showtime = showtime
        self.seat_number = seat_number.upper()
        self.seat_type = seat_type
        self.booking_time = datetime.now()
        self.status = "Pending"
        # เรียกใช้ Helper Function คิดราคา
        self.net_price = calculate_booking_price(showtime.base_price, seat_type, customer.member_type)

    def check_timeout(self):
        if self.status == "Pending":
            if datetime.now() > self.booking_time + timedelta(minutes=10):
                self.status = "Cancelled"
                return True
        return False

    def confirm_payment(self):
        if self.check_timeout():
            raise Exception("เกินเวลา 10 นาที (Timeout Auto-Cancel)")
        self.status = "Paid"
        self.customer.add_points(self.net_price)
        return self.net_price

# Helper Functions
def calculate_booking_price(base_price, seat_type="Normal", member_type="General", promo_discount=0.0):
    """ฟังก์ชันที่ 1: คำนวณราคาสุทธิ (มี Default Argument)"""
    seat_extras = {"Normal": 0, "Deluxe": 40, "Honeymoon": 80, "VIP": 190}
    member_discounts = {"General": 0.0, "Student": 0.20, "VIP": 0.30}

    total_seat_price = base_price + seat_extras.get(seat_type, 0)
    best_discount = max(member_discounts.get(member_type, 0.0), promo_discount)
    return round(total_seat_price * (1 - best_discount), 2)


def validate_seat_booking(seat_number, existing_seats_list, valid_seat_range=None):
    """ฟังก์ชันที่ 2: ตรวจสอบผังและที่นั่งซ้ำ (มี Default Argument)"""
    if valid_seat_range is None:
        valid_seat_range = [f"{row}{num}" for row in "ABCDEF" for num in range(1, 11)]

    seat_upper = seat_number.upper()
    if seat_upper not in valid_seat_range:
        return False, f"ที่นั่ง {seat_number} ไม่มีอยู่ในระบบ (Invalid Seat)"
    if seat_upper in existing_seats_list:
        return False, f"ที่นั่ง {seat_number} ถูกจองแล้ว (Double Booking)"
    return True, "ที่นั่งว่าง สามารถจองได้"


def process_ticket_scan(ticket_id, current_status="Paid", auto_update=True):
    """ฟังก์ชันที่ 3: จำลองการสแกน QR Code เข้าโรง (มี Default Argument)"""
    if current_status == "Used":
        return False, f"ตั๋ว {ticket_id} ถูกใช้งานไปแล้ว ไม่สามารถสแกนซ้ำได้!"
    elif current_status != "Paid":
        return False, f"ตั๋ว {ticket_id} ยังไม่ได้ชำระเงิน"

    new_status = "Used" if auto_update else current_status
    return True, f"สแกนสำเร็จ! ยินดีต้อนรับเข้าสู่โรงภาพยนตร์ ({new_status})"



# Execution Simulation Loop (จำลอง 300 รายการทีละคน)

showtime_1 = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 14, 0),
    datetime(2026, 8, 20, 17, 0),
    160,
)
showtime_2 = Showtime(
    "ST02",
    "Spider-Man",
    2,
    datetime(2026, 8, 20, 15, 30),
    datetime(2026, 8, 20, 18, 0),
    180,
)
showtime_3 = Showtime(
    "ST03",
    "Batman",
    3,
    datetime(2026, 8, 20, 19, 0),
    datetime(2026, 8, 20, 22, 0),
    200,
)

showtimes_pool = [showtime_1, showtime_2, showtime_3]

# แยกเก็บรายการที่นั่งที่ถูกจองแล้วของแต่ละโรง (ป้องกันการจองชนกันข้ามโรง)
booked_seats_by_theater = {1: [], 2: [], 3: []}

all_bookings = []
member_pool = ["General", "Student", "VIP"]
seat_type_pool = ["Normal", "Deluxe", "Honeymoon", "VIP"]
seat_pool = [f"{r}{n}" for r in "ABCDEF" for n in range(1, 11)] + ["Z99"]

# Loop จำลอง 300 รายการ (สุ่มเลือกโรงภาพยนตร์)
for i in range(1, 301):
    cust_id = f"C{i:03d}"
    cust_name = f"Customer_{i}"
    chosen_member = random.choice(member_pool)
    chosen_seat = random.choice(seat_pool)
    chosen_seat_type = random.choice(seat_type_pool)
    booking_id = f"BK{i:03d}"

    # สุ่มเลือกโรงภาพยนตร์ 1 จาก 3 โรง
    chosen_showtime = random.choice(showtimes_pool)
    theater_no = chosen_showtime.theater_no

    customer = Customer(cust_id, cust_name, chosen_member)

    # ตรวจสอบที่นั่งซ้ำเฉพาะในโรงภาพยนตร์ที่เลือก
    is_valid, seat_msg = validate_seat_booking(
        chosen_seat, booked_seats_by_theater[theater_no]
    )

    if not is_valid:
        failed_booking = Booking(
            booking_id,
            customer,
            chosen_showtime,
            chosen_seat,
            chosen_seat_type,
        )
        failed_booking.status = f"Rejected ({seat_msg})"
        failed_booking.net_price = 0.0
        all_bookings.append(failed_booking)
        continue

    try:
        new_booking = Booking(
            booking_id,
            customer,
            chosen_showtime,
            chosen_seat,
            chosen_seat_type,
        )

        if random.random() < 0.10:
            new_booking.booking_time = datetime.now() - timedelta(minutes=15)

        new_booking.confirm_payment()
        booked_seats_by_theater[theater_no].append(
            chosen_seat
        )
        all_bookings.append(new_booking)

    except Exception:
        new_booking.net_price = 0.0
        all_bookings.append(new_booking)


## ส่วนที่ 2 — ทดสอบฟังก์ชันทีละตัว

In [4]:
seat_pool = [f"{r}{n}" for r in "ABCDEF" for n in range(1, 11)]
# ทดสอบเรียกสุ่มที่นั่ง 3 ครั้ง ดูว่าฟังก์ชันสุ่มทำงานได้ค่าไม่ซ้ำกันในแต่ละรอบ
for _ in range(3):
    print("ที่นั่งที่สุ่มได้:", random.choice(seat_pool))

ที่นั่งที่สุ่มได้: C4
ที่นั่งที่สุ่มได้: E1
ที่นั่งที่สุ่มได้: B8


In [5]:
booked_test_list = ["A1"]

# ทดสอบการตรวจที่นั่ง (ที่นั่งซ้ำ, ที่นั่งว่าง, ที่นั่งไม่มีอยู่จริง)
for seat_to_check in ["A1", "B5", "Z99"]:
    is_valid, message = validate_seat_booking(seat_to_check, booked_test_list)
    print(f"ผลการตรวจที่นั่ง '{seat_to_check}':", message)

ผลการตรวจที่นั่ง 'A1': ที่นั่ง A1 ถูกจองแล้ว (Double Booking)
ผลการตรวจที่นั่ง 'B5': ที่นั่งว่าง สามารถจองได้
ผลการตรวจที่นั่ง 'Z99': ที่นั่ง Z99 ไม่มีอยู่ในระบบ (Invalid Seat)


In [6]:
# เปรียบเทียบการเรียก calculate_booking_price() แบบไม่ระบุค่า (ใช้ default) กับแบบระบุช่วงเอง
print("เรียกแบบ default (ไม่ใส่ argument):", calculate_booking_price(160))
print(
    "เรียกแบบระบุช่วงเอง (ที่นั่ง VIP, สมาชิก VIP):",
    calculate_booking_price(160, "VIP", "VIP"),
)
print(
    "เรียกแบบระบุแค่ promo_discount (keyword argument):",
    calculate_booking_price(160, promo_discount=0.50),
)

เรียกแบบ default (ไม่ใส่ argument): 160.0
เรียกแบบระบุช่วงเอง (ที่นั่ง VIP, สมาชิก VIP): 245.0
เรียกแบบระบุแค่ promo_discount (keyword argument): 80.0


## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา

In [7]:
def explain_booking_price(booking):
    """รับ object Booking 1 ตัว -> พิมพ์อธิบายการคำนวณราคาทีละขั้นตอน"""
    seat_extras = {"Normal": 0, "Deluxe": 40, "Honeymoon": 80, "VIP": 190}

    extra_price = seat_extras.get(booking.seat_type, 0)
    base_price = booking.showtime.base_price
    full_price = base_price + extra_price

    discount_rate = booking.customer.get_discount_rate()

    final_price = booking.net_price

    print(
        f"ราคาตั๋วภาพยนตร์เรื่อง '{booking.showtime.movie_name}' (โรง"
        f" {booking.showtime.theater_no}) = {base_price} บาท"
    )
    print(
        f"บวกเพิ่มที่นั่งประเภท '{booking.seat_type}' = +{extra_price} บาท"
        f" (ราคารวมก่อนส่วนลด = {full_price} บาท)"
    )

    if discount_rate > 0:
        print(
            f"ลูกค้าเป็นสมาชิก {booking.customer.member_type} -> ได้ส่วนลด"
            f" {discount_rate * 100:.0f}%"
        )
    else:
        print("ลูกค้าเป็นสมาชิก General -> ไม่มีส่วนลด")

    print(f"ราคาสุทธิ = {final_price:,.2f} บาท")
    return


demo_cust = Customer("C001", "Customer_1", "VIP")
demo_show = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 18, 0),
    datetime(2026, 8, 20, 21, 0),
    160,
)
demo_booking = Booking("BK001", demo_cust, demo_show, "A1", "Honeymoon")
explain_booking_price(demo_booking)

ราคาตั๋วภาพยนตร์เรื่อง 'Avatar 3' (โรง 1) = 160 บาท
บวกเพิ่มที่นั่งประเภท 'Honeymoon' = +80 บาท (ราคารวมก่อนส่วนลด = 240 บาท)
ลูกค้าเป็นสมาชิก VIP -> ได้ส่วนลด 30%
ราคาสุทธิ = 168.00 บาท


## ตารางบันทึกข้อมูลที่จำลองได้

In [8]:
summary_rows = [
    {
        "booking_id": b.booking_id,
        "customer_id": b.customer.customer_id,
        "name": b.customer.name,
        "member_type": b.customer.member_type,
        "movie_name": b.showtime.movie_name,
        "theater_no": b.showtime.theater_no,
        "seat_number": b.seat_number,
        "seat_type": b.seat_type,
        "price": b.net_price,
        "status": b.status,
    }
    for b in all_bookings
]

demo_summary_df = pd.DataFrame(summary_rows)
demo_summary_df

,booking_id,customer_id,name,member_type,movie_name,theater_no,seat_number,seat_type,price,status
0,BK001,C001,Customer_1,VIP,Batman,3,A3,Honeymoon,196.0,Paid
1,BK002,C002,Customer_2,General,Spider-Man,2,B1,Normal,180.0,Paid
2,BK003,C003,Customer_3,VIP,Avatar 3,1,C6,VIP,245.0,Paid
3,BK004,C004,Customer_4,General,Spider-Man,2,C8,Normal,180.0,Paid
4,BK005,C005,Customer_5,VIP,Avatar 3,1,F2,Normal,112.0,Paid
...,...,...,...,...,...,...,...,...,...,...
295,BK296,C296,Customer_296,VIP,Spider-Man,2,D5,Deluxe,0.0,Rejected (ที่นั่ง D5 ถูกจองแล้ว (Double Booking))
296,BK297,C297,Customer_297,General,Spider-Man,2,A5,VIP,0.0,Rejected (ที่นั่ง A5 ถูกจองแล้ว (Double Booking))
297,BK298,C298,Customer_298,General,Spider-Man,2,A8,Honeymoon,260.0,Paid
298,BK299,C299,Customer_299,Student,Avatar 3,1,E4,VIP,0.0,Rejected (ที่นั่ง E4 ถูกจองแล้ว (Double Booking))


In [9]:
demo_summary_df.to_csv("demo_cinema.csv")

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
demo_summary_df.to_csv("/content/drive/MyDrive/demo_cinema.csv", index=False)

## ฐานข้อมูล SQLite

In [12]:
import sqlite3
import pandas as pd

# เชื่อมต่อฐานข้อมูล SQLite
conn = sqlite3.connect("cinema.db")

time_slots = ["11:00", "13:30", "16:00", "18:30", "21:00"]

# ตารางที่ 1: customers (ตารางลูกค้า - ดึงเฉพาะข้อมูลลูกค้าไม่ให้ซ้ำ)
customers_list = [
    {
        "customer_id": b.customer.customer_id,
        "name": b.customer.name,
        "member_type": b.customer.member_type,
        "points": b.customer.points,
    }
    for b in all_bookings
]
# ใช้ .drop_duplicates() เพื่อให้เหลือลูกค้า 1 คน ต่อ 1 แถวในตารางแม่
customers_df = pd.DataFrame(customers_list).drop_duplicates(
    subset=["customer_id"]
)

# บันทึกลงตาราง customers
customers_df.to_sql("customers", conn, if_exists="replace", index=False)


# ตารางที่ 2: bookings (ตารางธุรกรรมการจอง - ผูกด้วย customer_id)
bookings_list = [
    {
        "booking_id": b.booking_id,
        "customer_id": b.customer.customer_id,  # ใช้เป็น Foreign Key เชื่อมกับตารางแม่
        "movie_name": b.showtime.movie_name,
        "showtime": random.choice(time_slots),
        "theater_no": b.showtime.theater_no,
        "seat_number": b.seat_number,
        "seat_type": b.seat_type,
        "price": b.net_price,
        "status": b.status,
    }
    for b in all_bookings
]

bookings_df = pd.DataFrame(bookings_list)

# บันทึกลงตาราง bookings
bookings_df.to_sql("bookings", conn, if_exists="replace", index=False)


# ทดสอบดึงข้อมูลดูว่าเข้าฐานข้อมูลจริงไหม
check_customers = pd.read_sql_query(
    "SELECT * FROM customers LIMIT 5", conn
)
check_bookings = pd.read_sql_query("SELECT * FROM bookings LIMIT 5", conn)

print("--- ตารางที่ 1: customers ---")
print(check_customers)
print("\n--- ตารางที่ 2: bookings ---")
print(check_bookings)

--- ตารางที่ 1: customers ---
  customer_id        name member_type  points
0        C001  Customer_1         VIP      19
1        C002  Customer_2     General      18
2        C003  Customer_3         VIP      24
3        C004  Customer_4     General      18
4        C005  Customer_5         VIP      11

--- ตารางที่ 2: bookings ---
  booking_id customer_id  movie_name showtime  theater_no seat_number  \
0      BK001        C001      Batman    18:30           3          A3   
1      BK002        C002  Spider-Man    11:00           2          B1   
2      BK003        C003    Avatar 3    16:00           1          C6   
3      BK004        C004  Spider-Man    11:00           2          C8   
4      BK005        C005    Avatar 3    18:30           1          F2   

   seat_type  price status  
0  Honeymoon  196.0   Paid  
1     Normal  180.0   Paid  
2        VIP  245.0   Paid  
3     Normal  180.0   Paid  
4     Normal  112.0   Paid  


In [13]:
query_join = """
SELECT
    b.booking_id,
    c.name AS customer_name,
    c.member_type,
    b.movie_name,
    b.showtime,
    b.theater_no,
    b.seat_number,
    b.price,
    b.status
FROM bookings b
JOIN customers c ON b.customer_id = c.customer_id
LIMIT 5
"""

# ดึงข้อมูลออกมาเป็น DataFrame
df_joined = pd.read_sql_query(query_join, conn)
print(df_joined)

# ปิด connection หลังใช้งานเสร็จ
conn.close()

  booking_id customer_name member_type  movie_name showtime  theater_no  \
0      BK001    Customer_1         VIP      Batman    18:30           3   
1      BK002    Customer_2     General  Spider-Man    11:00           2   
2      BK003    Customer_3         VIP    Avatar 3    16:00           1   
3      BK004    Customer_4     General  Spider-Man    11:00           2   
4      BK005    Customer_5         VIP    Avatar 3    18:30           1   

  seat_number  price status  
0          A3  196.0   Paid  
1          B1  180.0   Paid  
2          C6  245.0   Paid  
3          C8  180.0   Paid  
4          F2  112.0   Paid  


## ส่วนที่ 4

In [14]:
def simulate_customer_visit(
    customer,
    showtime,
    seat_number,
    seat_type,
    existing_seats,
    promo_discount=0.0
):

    print("" * 40)
    print("เริ่มกระบวนการจองตั๋วภาพยนตร์")
    print("" * 40)

    # Step 1 ลูกค้าเข้ามา
    print("\n ลูกค้าเดินเข้ามาจองตั๋ว")
    print(f"รหัสลูกค้า: {customer.customer_id}")
    print(f"ชื่อลูกค้า: {customer.name}")
    print(f"ประเภทสมาชิก: {customer.member_type}")

    # Step 2 เลือกรอบหนัง
    print("\n เลือกรอบภาพยนตร์")
    print(f"ภาพยนตร์: {showtime.movie_name}")
    print(f"โรงภาพยนตร์: {showtime.theater_no}")
    print(f"เวลาเริ่ม: {showtime.start_time}")

    # Step 3 ตรวจสอบที่นั่ง
    print("\n ตรวจสอบที่นั่ง")

    is_valid, seat_msg = validate_seat_booking(
        seat_number,
        existing_seats
    )

    print(seat_msg)

    if not is_valid:
        print("\nไม่สามารถทำรายการจองได้")
        return None

    # Step 4 คำนวณราคา
    print("\n คำนวณราคาตั๋ว")

    seat_extras = {
        "Normal": 0,
        "Deluxe": 40,
        "Honeymoon": 80,
        "VIP": 190
    }

    # ราคาปกติ = ราคาตั๋วพื้นฐาน + ค่าเพิ่มประเภทที่นั่ง
    normal_price = (
        showtime.base_price +
        seat_extras.get(seat_type, 0)
    )

    # ราคาสุทธิหลังหักส่วนลด
    net_price = calculate_booking_price(
        showtime.base_price,
        seat_type,
        customer.member_type,
        promo_discount
    )

    print(f"ประเภทที่นั่ง: {seat_type}")
    print(f"ราคาปกติ: {normal_price} บาท")
    print(f"ราคาสุทธิ: {net_price} บาท")

    # Step 5 สร้าง Booking
    print("\n สร้างรายการจอง")

    booking_id = f"BK{customer.customer_id}"

    booking = Booking(
        booking_id,
        customer,
        showtime,
        seat_number,
        seat_type
    )

    # ใช้ราคาที่คำนวณจากโปรโมชั่น
    booking.net_price = net_price

    print("สร้างรายการจองสำเร็จ")
    print(f"Booking ID: {booking.booking_id}")
    print(f"สถานะ: {booking.status}")

    # Step 6 ชำระเงิน
    print("\n ลูกค้าชำระเงิน")

    try:
        paid_price = booking.confirm_payment()

        print(f"ชำระเงินสำเร็จ: {paid_price} บาท")
        print(f"สถานะปัจจุบัน: {booking.status}")

        # เพิ่มที่นั่งเข้าไปในรายการที่จองแล้ว
        existing_seats.append(seat_number.upper())

    except Exception as e:
        print(f"ชำระเงินไม่สำเร็จ: {e}")
        return booking

    # Step 7 สะสมคะแนน
    print("\n สะสมคะแนนสมาชิก")
    print(f"คะแนนสะสมปัจจุบัน: {customer.points} คะแนน")

    # Step 8 สแกนตั๋ว
    print("\n สแกนตั๋วเข้าโรงภาพยนตร์")

    success, scan_msg = process_ticket_scan(
        booking.booking_id,
        booking.status
    )

    print(scan_msg)

    # สรุปผล
    print("\n" + "" * 40)
    print("สรุปการใช้บริการ")
    print("" * 40)

    print(f"ลูกค้า: {customer.name}")
    print(f"ภาพยนตร์: {showtime.movie_name}")
    print(f"โรงภาพยนตร์: {showtime.theater_no}")
    print(f"ที่นั่ง: {booking.seat_number}")
    print(f"ประเภทที่นั่ง: {booking.seat_type}")
    print(f"ราคาปกติ: {normal_price} บาท")
    print(f"ราคาสุทธิ: {booking.net_price} บาท")
    print(f"สถานะการจอง: {booking.status}")
    print(f"คะแนนสะสม: {customer.points} คะแนน")

    return booking

In [15]:
customer_1 = Customer(
    customer_id="C001",
    name="customer_n",
    member_type="Student"
)

showtime_1 = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 14, 0),
    datetime(2026, 8, 20, 17, 0),
    160
)

existing_seats = ["A1", "A2", "B5"]

booking_result = simulate_customer_visit(
    customer=customer_1,
    showtime=showtime_1,
    seat_number="C3",
    seat_type="Deluxe",
    existing_seats=existing_seats,
    promo_discount=0.10
)


เริ่มกระบวนการจองตั๋วภาพยนตร์


 ลูกค้าเดินเข้ามาจองตั๋ว
รหัสลูกค้า: C001
ชื่อลูกค้า: customer_n
ประเภทสมาชิก: Student

 เลือกรอบภาพยนตร์
ภาพยนตร์: Avatar 3
โรงภาพยนตร์: 1
เวลาเริ่ม: 2026-08-20 14:00:00

 ตรวจสอบที่นั่ง
ที่นั่งว่าง สามารถจองได้

 คำนวณราคาตั๋ว
ประเภทที่นั่ง: Deluxe
ราคาปกติ: 200 บาท
ราคาสุทธิ: 160.0 บาท

 สร้างรายการจอง
สร้างรายการจองสำเร็จ
Booking ID: BKC001
สถานะ: Pending

 ลูกค้าชำระเงิน
ชำระเงินสำเร็จ: 160.0 บาท
สถานะปัจจุบัน: Paid

 สะสมคะแนนสมาชิก
คะแนนสะสมปัจจุบัน: 16 คะแนน

 สแกนตั๋วเข้าโรงภาพยนตร์
สแกนสำเร็จ! ยินดีต้อนรับเข้าสู่โรงภาพยนตร์ (Used)


สรุปการใช้บริการ

ลูกค้า: customer_n
ภาพยนตร์: Avatar 3
โรงภาพยนตร์: 1
ที่นั่ง: C3
ประเภทที่นั่ง: Deluxe
ราคาปกติ: 200 บาท
ราคาสุทธิ: 160.0 บาท
สถานะการจอง: Paid
คะแนนสะสม: 16 คะแนน


## ส่วนที่ 5

In [16]:
import random

# สร้างรอบหนังหลายรอบ
showtime_1 = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 14, 0),
    datetime(2026, 8, 20, 17, 0),
    160
)

showtime_2 = Showtime(
    "ST02",
    "Spider-Man",
    2,
    datetime(2026, 8, 20, 15, 30),
    datetime(2026, 8, 20, 18, 0),
    180
)

showtime_3 = Showtime(
    "ST03",
    "Batman",
    3,
    datetime(2026, 8, 20, 19, 0),
    datetime(2026, 8, 20, 22, 0),
    200
)

# รวมรอบหนังไว้สำหรับสุ่มเลือก
showtimes_pool = [
    showtime_1,
    showtime_2,
    showtime_3
]

# เก็บที่นั่งที่ถูกจองแยกตามโรง
booked_seats_by_theater = {
    1: [],
    2: [],
    3: []
}

# ข้อมูลสำหรับสุ่ม
member_pool = ["General", "Student", "VIP"]

seat_type_pool = [
    "Normal",
    "Deluxe",
    "Honeymoon",
    "VIP"
]

seat_pool = [
    f"{row}{num}"
    for row in "ABCDEF"
    for num in range(1, 11)
] + ["Z99"]


for i in range(1, 6):

    print("\n")
    print("" * 50)
    print(f"ลูกค้าคนที่ {i}")
    print("" * 50)

    # สุ่มประเภทสมาชิก
    chosen_member = random.choice(member_pool)

    # สุ่มรอบหนัง
    chosen_showtime = random.choice(showtimes_pool)

    # สุ่มที่นั่ง
    chosen_seat = random.choice(seat_pool)

    # สุ่มประเภทที่นั่ง
    chosen_seat_type = random.choice(
        seat_type_pool
    )

    # สร้างลูกค้า
    customer = Customer(
        customer_id=f"C{i:03d}",
        name=f"Customer_{i}",
        member_type=chosen_member
    )

    # เรียกใช้ฟังก์ชันเดิม
    booking_result = simulate_customer_visit(
        customer=customer,
        showtime=chosen_showtime,
        seat_number=chosen_seat,
        seat_type=chosen_seat_type,

        # ส่งรายการที่นั่งของโรงที่ลูกค้าเลือก
        existing_seats=
        booked_seats_by_theater[
            chosen_showtime.theater_no
        ],

        # สุ่มโปรโมชั่น 0% หรือ 10%
        promo_discount=random.choice([
            0.0,
            0.10
        ])
    )




ลูกค้าคนที่ 1


เริ่มกระบวนการจองตั๋วภาพยนตร์


 ลูกค้าเดินเข้ามาจองตั๋ว
รหัสลูกค้า: C001
ชื่อลูกค้า: Customer_1
ประเภทสมาชิก: VIP

 เลือกรอบภาพยนตร์
ภาพยนตร์: Spider-Man
โรงภาพยนตร์: 2
เวลาเริ่ม: 2026-08-20 15:30:00

 ตรวจสอบที่นั่ง
ที่นั่งว่าง สามารถจองได้

 คำนวณราคาตั๋ว
ประเภทที่นั่ง: Honeymoon
ราคาปกติ: 260 บาท
ราคาสุทธิ: 182.0 บาท

 สร้างรายการจอง
สร้างรายการจองสำเร็จ
Booking ID: BKC001
สถานะ: Pending

 ลูกค้าชำระเงิน
ชำระเงินสำเร็จ: 182.0 บาท
สถานะปัจจุบัน: Paid

 สะสมคะแนนสมาชิก
คะแนนสะสมปัจจุบัน: 18 คะแนน

 สแกนตั๋วเข้าโรงภาพยนตร์
สแกนสำเร็จ! ยินดีต้อนรับเข้าสู่โรงภาพยนตร์ (Used)


สรุปการใช้บริการ

ลูกค้า: Customer_1
ภาพยนตร์: Spider-Man
โรงภาพยนตร์: 2
ที่นั่ง: A4
ประเภทที่นั่ง: Honeymoon
ราคาปกติ: 260 บาท
ราคาสุทธิ: 182.0 บาท
สถานะการจอง: Paid
คะแนนสะสม: 18 คะแนน



ลูกค้าคนที่ 2


เริ่มกระบวนการจองตั๋วภาพยนตร์


 ลูกค้าเดินเข้ามาจองตั๋ว
รหัสลูกค้า: C002
ชื่อลูกค้า: Customer_2
ประเภทสมาชิก: Student

 เลือกรอบภาพยนตร์
ภาพยนตร์: Avatar 3
โรงภาพยนตร์: 1
เวลาเริ่ม: 2026-08-20